# MobileI2V — Test SD-Turbo sur Google Colab
Ce notebook génère des images réellement calculées par un modèle de diffusion. **Ce n'est pas encore une animation cohérente garantie.** Exécution GPU gratuite selon disponibilité Colab. Aucune API payante.
1. Dans **Exécution > Modifier le type d'exécution**, sélectionner GPU T4 si proposé.
2. Exécuter les cellules dans l'ordre, importer une photo, saisir une consigne.
3. Télécharger le ZIP des images et examiner la continuité.


In [ ]:
%pip -q install 'diffusers>=0.32,<1' 'transformers>=4.44,<5' 'accelerate>=0.33,<2' 'safetensors>=0.4,<1' 'pillow>=10,<13'
import torch
assert torch.cuda.is_available(), 'GPU indisponible : choisir GPU T4 dans les paramètres Colab ou réessayer ultérieurement.'
print('GPU :', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
from PIL import Image
import io, re, zipfile, os
from diffusers import AutoPipelineForImage2Image
uploaded = files.upload()
assert uploaded, 'Importer une photo pour continuer'
filename = next(iter(uploaded))
reference = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')
reference.thumbnail((512,512), Image.Resampling.LANCZOS)
w = max(64, round(reference.width/8)*8)
h = max(64, round(reference.height/8)*8)
reference = reference.resize((w,h),Image.Resampling.LANCZOS)
display(reference)


In [ ]:
# Modifier uniquement ces paramètres pour le premier essai.
histoire = 'Le personnage marche puis court puis tombe au sol'
nombre_images = 6  # Commencer petit pour économiser le quota GPU
force_reference = 0.85
assert 2 <= nombre_images <= 24
model = AutoPipelineForImage2Image.from_pretrained('stabilityai/sd-turbo', torch_dtype=torch.float16)
model.to('cuda')
model.set_progress_bar_config(disable=True)
print('Modèle chargé')


In [ ]:
def action_at(story, position):
    phases = [x.strip() for x in re.split(r'(?i)\b(?:et puis|puis|ensuite|then|afterwards|suddenly)\b|[.;\n]',story) if x.strip()]
    return phases[min(len(phases)-1,int(min(position,0.999999)*len(phases)))] if phases else story

images = []
previous = reference
for i in range(nombre_images):
    phase = i/nombre_images
    current_action = action_at(histoire,phase)
    anchor = min(0.4,max(0.1,force_reference*0.35))
    input_image = Image.blend(previous,reference,anchor)
    prompt = f'{histoire}. Current action: {current_action}. Keyframe {i+1}/{nombre_images}. Same person, same clothes, coherent anatomy, changed body pose.'
    with torch.inference_mode():
        image = model(prompt=prompt,image=input_image,strength=0.55,guidance_scale=0.0,num_inference_steps=2).images[0]
    images.append(image)
    previous = image
    print(f'Image {i+1}/{nombre_images}: {current_action}')
    display(image.resize((min(256,image.width),round(image.height*min(256,image.width)/image.width))))


In [ ]:
from IPython.display import display
archive_name = 'mobilei2v_images_ia.zip'
with zipfile.ZipFile(archive_name,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for i,img in enumerate(images):
        b=io.BytesIO();img.save(b,format='PNG');z.writestr(f'frame_{i:04d}.png',b.getvalue())
    z.writestr('prompt.txt',histoire)
print('Images enregistrées :',len(images))
files.download(archive_name)
